# Week 6.2 — PageRank by the Power Method
Graph: `1 -> 2, 2 -> 3, 3 -> 1 and 2` (3-node toy example with a residual check).

In [1]:
import numpy as np
import matplotlib.pyplot as plt

## Build the Google matrix
$P$ is a column-stochastic link matrix for the 3-node toy graph. The Google matrix $G=\alpha P + \frac{1-\alpha}{n}\mathbf{1}\mathbf{1}^T$ mixes in a uniform "teleportation" term with damping factor $\alpha=0.85$, which makes $G$ a primitive stochastic matrix — guaranteeing a unique dominant eigenvalue $1$ with a positive eigenvector (the PageRank vector).

In [2]:
P = np.array([[0, 0, 0.5],
              [1, 0, 0.5],
              [0, 1, 0]])  # column-stochastic
alpha = 0.85
n = P.shape[0]
G = alpha * P + (1 - alpha) / n * np.ones((n, n))  # Google matrix

## Power iteration
Repeatedly applies $x^{k+1}=Gx^k$; since $G$ is column-stochastic, no renormalization is needed. Because the dominant eigenvalue $1$ is separated from the rest of the spectrum by the damping factor $\alpha$, this power iteration converges quickly to the PageRank vector, tracked here via the $\ell_1$ change between successive iterates.

In [3]:
x = np.ones(n) / n  # initial rank vector
tol = 1e-12
maxit = 200
res = []

for k in range(maxit):
    x_new = G @ x  # power step (no need to renormalize; G is stochastic)
    residual = np.linalg.norm(x_new - x, 1)
    res.append(residual)
    x = x_new
    if residual < tol:
        break

print('PageRank vector (l1-normalized):')
x = x / np.sum(x)
print(x)
print(f'Iterations: {len(res)}, ||x_{{k+1}}-x_k||_1 = {res[-1]:.2e}')

PageRank vector (l1-normalized):
[0.21481063 0.39739966 0.38778971]
Iterations: 53, ||x_{k+1}-x_k||_1 = 9.02e-13


## Convergence plot
Shows the $\ell_1$ change per iteration decaying geometrically, at the rate governed by $\alpha$ (the ratio of the two largest eigenvalue magnitudes of $G$).

In [4]:
plt.figure()
plt.semilogy(np.arange(1, len(res) + 1), res, 'o-')
plt.grid(True)
plt.xlabel('Iteration')
plt.ylabel('||x_{k+1}-x_k||_1')
plt.title('Power method for PageRank (3-node toy)')
plt.show()

/var/folders/k6/1w07pxzj0mx129drg82_k3_w0000gp/T/ipykernel_73682/4056664346.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## (Optional) Compare to eigenvector solution
Cross-checks the power-iteration result against the eigenvector of $G$ corresponding to the eigenvalue closest to $1$, computed directly via a full eigendecomposition — confirming power iteration converged to the right answer.

In [5]:
eigvals, eigvecs = np.linalg.eig(G)
idx = np.argmax(np.real(eigvals))  # Find eigenvector for eigenvalue near 1
x_eig = np.real(eigvecs[:, idx])
x_eig = x_eig / np.sum(x_eig)

print(f'||x_power - x_eig||_1 = {np.linalg.norm(x - x_eig, 1):.2e}')

||x_power - x_eig||_1 = 4.94e-13
